# Required Libraries

In [1]:
# All libraries here

# Basic libraries
import os
import time
import pprint
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import lognorm
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from datetime import datetime, timedelta

# Sklearn libraries
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Tensorflow libraries
from tensorflow.keras import Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential, load_model

# Original Demographic Specific Dataset

In [11]:
# Loading the demographic specific dataset
master_df = pd.read_csv(os.path.join('final_master_datafile.csv'))

# Data Acquisition

In [12]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 554400 entries, 0 to 554399
Data columns (total 28 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   datetime                    554400 non-null  str    
 1   athlete                     554400 non-null  int64  
 2   block_id                    554400 non-null  str    
 3   distance                    554400 non-null  float64
 4   duration                    554400 non-null  float64
 5   days_until_marathon         554400 non-null  int64  
 6   marathon_count_until_2019   554400 non-null  int64  
 7   marathon_count_2019         554400 non-null  int64  
 8   avg_activity_pace           554400 non-null  float64
 9   cumulative_duration         554400 non-null  float64
 10  cumulative_distance         554400 non-null  float64
 11  performance_category        554400 non-null  int64  
 12  fastest_pace                554400 non-null  float64
 13  slowest_pace             

In [13]:
master_df.head(10)

,datetime,athlete,block_id,distance,duration,days_until_marathon,marathon_count_until_2019,marathon_count_2019,avg_activity_pace,cumulative_duration,...,min_distance,max_distance,active_streak,rest_streak,duration_last_7_days,distance_last_7_days,weekly_cumulative_duration,weekly_cumulative_distance,avg_race_pace,marathon_race_time
0,2018-12-24,15,15_boston,0.00,0.0,-112,2,1,0.000000,0.0,...,0.00,0.00,0,1,0.0,0.00,0.0,0.00,4.661017,198.0
1,2018-12-25,15,15_boston,0.00,0.0,-111,2,1,0.000000,0.0,...,0.00,0.00,0,2,0.0,0.00,0.0,0.00,4.661017,198.0
2,2018-12-26,15,15_boston,0.00,0.0,-110,2,1,0.000000,0.0,...,0.00,0.00,0,3,0.0,0.00,0.0,0.00,4.661017,198.0
3,2018-12-27,15,15_boston,0.00,0.0,-109,2,1,0.000000,0.0,...,0.00,0.00,0,4,0.0,0.00,0.0,0.00,4.661017,198.0
4,2018-12-28,15,15_boston,0.00,0.0,-108,2,1,0.000000,0.0,...,0.00,0.00,0,5,0.0,0.00,0.0,0.00,4.661017,198.0
5,2018-12-29,15,15_boston,0.00,0.0,-107,2,1,0.000000,0.0,...,0.00,0.00,0,6,0.0,0.00,0.0,0.00,4.661017,198.0
6,2018-12-30,15,15_boston,0.00,0.0,-106,2,1,0.000000,0.0,...,0.00,0.00,0,7,0.0,0.00,0.0,0.00,4.661017,198.0
7,2018-12-31,15,15_boston,0.00,0.0,-105,2,1,0.000000,0.0,...,0.00,0.00,0,8,0.0,0.00,0.0,0.00,4.661017,198.0
8,2019-01-01,15,15_boston,19.49,95.0,-104,2,1,4.874295,95.0,...,19.49,19.49,1,0,95.0,19.49,95.0,19.49,4.661017,198.0
9,2019-01-02,15,15_boston,2.54,24.3,-103,2,1,9.566929,119.3,...,2.54,19.49,2,0,119.3,22.03,119.3,22.03,4.661017,198.0


In [14]:
master_df.tail(10)

,datetime,athlete,block_id,distance,duration,days_until_marathon,marathon_count_until_2019,marathon_count_2019,avg_activity_pace,cumulative_duration,...,min_distance,max_distance,active_streak,rest_streak,duration_last_7_days,distance_last_7_days,weekly_cumulative_duration,weekly_cumulative_distance,avg_race_pace,marathon_race_time
554390,2019-10-24,37520,37520_new_york,0.00,0.0,-10,1,1,0.000000,1093.75,...,1.76,21.09,0,1,174.10,42.72,92.1,21.63,4.668074,199.0
554391,2019-10-25,37520,37520_new_york,0.00,0.0,-9,1,1,0.000000,1093.75,...,1.76,21.09,0,2,174.10,42.72,92.1,21.63,4.668074,199.0
554392,2019-10-26,37520,37520_new_york,0.00,0.0,-8,1,1,0.000000,1093.75,...,1.76,21.09,0,3,92.10,21.63,92.1,21.63,4.668074,199.0
554393,2019-10-27,37520,37520_new_york,0.00,0.0,-7,1,1,0.000000,1093.75,...,1.76,21.09,0,4,92.10,21.63,0.0,0.00,4.668074,199.0
554394,2019-10-28,37520,37520_new_york,0.00,0.0,-6,1,1,0.000000,1093.75,...,1.76,21.09,0,5,42.35,9.72,0.0,0.00,4.668074,199.0
554395,2019-10-29,37520,37520_new_york,0.00,0.0,-5,1,1,0.000000,1093.75,...,1.76,21.09,0,6,42.35,9.72,0.0,0.00,4.668074,199.0
554396,2019-10-30,37520,37520_new_york,15.02,68.0,-4,1,1,4.527297,1161.75,...,1.76,21.09,1,0,68.00,15.02,68.0,15.02,4.668074,199.0
554397,2019-10-31,37520,37520_new_york,0.00,0.0,-3,1,1,0.000000,1161.75,...,1.76,21.09,0,1,68.00,15.02,68.0,15.02,4.668074,199.0
554398,2019-11-01,37520,37520_new_york,0.00,0.0,-2,1,1,0.000000,1161.75,...,1.76,21.09,0,2,68.00,15.02,68.0,15.02,4.668074,199.0
554399,2019-11-02,37520,37520_new_york,0.00,0.0,-1,1,1,0.000000,1161.75,...,1.76,21.09,0,3,68.00,15.02,68.0,15.02,4.668074,199.0


In [15]:
master_df.isnull().sum()

datetime                      0
athlete                       0
block_id                      0
distance                      0
duration                      0
days_until_marathon           0
marathon_count_until_2019     0
marathon_count_2019           0
avg_activity_pace             0
cumulative_duration           0
cumulative_distance           0
performance_category          0
fastest_pace                  0
slowest_pace                  0
weekly_fastest_pace           0
weekly_slowest_pace           0
min_duration                  0
max_duration                  0
min_distance                  0
max_distance                  0
active_streak                 0
rest_streak                   0
duration_last_7_days          0
distance_last_7_days          0
weekly_cumulative_duration    0
weekly_cumulative_distance    0
avg_race_pace                 0
marathon_race_time            0
dtype: int64

In [16]:
master_df.isna().sum()

datetime                      0
athlete                       0
block_id                      0
distance                      0
duration                      0
days_until_marathon           0
marathon_count_until_2019     0
marathon_count_2019           0
avg_activity_pace             0
cumulative_duration           0
cumulative_distance           0
performance_category          0
fastest_pace                  0
slowest_pace                  0
weekly_fastest_pace           0
weekly_slowest_pace           0
min_duration                  0
max_duration                  0
min_distance                  0
max_distance                  0
active_streak                 0
rest_streak                   0
duration_last_7_days          0
distance_last_7_days          0
weekly_cumulative_duration    0
weekly_cumulative_distance    0
avg_race_pace                 0
marathon_race_time            0
dtype: int64

# Data Cleaning

Below is a list of unwanted features that will be dropped from the dataset. These were only relevant for sequential models as they show progression over time:
- cumulative_duration      
- cumulative_distance      
- performance_category
- weekly_fastest_pace        
- weekly_slowest_pace 
- duration_last_7_days      
- distance_last_7_days      
- weekly_cumulative_duration 
- weekly_cumulative_distance

In [21]:
# Removing any unwanted columns from the dataset
cleaned_master_df = master_df.copy()

features_to_drop = ['cumulative_duration', 'cumulative_distance', 'performance_category', 'weekly_fastest_pace', 'weekly_slowest_pace', 
                    'duration_last_7_days', 'distance_last_7_days', 'weekly_cumulative_duration', 'weekly_cumulative_distance']

cleaned_master_df.drop(features_to_drop, axis=1, inplace=True)

In [22]:
# Checking the list of features
cleaned_master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 554400 entries, 0 to 554399
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   datetime                   554400 non-null  str    
 1   athlete                    554400 non-null  int64  
 2   block_id                   554400 non-null  str    
 3   distance                   554400 non-null  float64
 4   duration                   554400 non-null  float64
 5   days_until_marathon        554400 non-null  int64  
 6   marathon_count_until_2019  554400 non-null  int64  
 7   marathon_count_2019        554400 non-null  int64  
 8   avg_activity_pace          554400 non-null  float64
 9   fastest_pace               554400 non-null  float64
 10  slowest_pace               554400 non-null  float64
 11  min_duration               554400 non-null  float64
 12  max_duration               554400 non-null  float64
 13  min_distance               554400 non-nu

# Feature Engineering & Aggregation

Below is a list of features that will either be kept for filtering or converted to an aggregated feature. The same process will be used to develop the timeframe reduced datasets (from 16 weeks to 8 weeks):
- datetime -> kept for possible filtering
- athlete -> kept for possible filtering                  
- block_id -> kept for possible filtering                 
- distance -> converted to: total_distance                 
- duration -> converted to: total_duration                 
- days_until_marathon -> kept for dataset volume reduction       
- marathon_count_until_2019 -> kept the same, not altered, shows experience
- marathon_count_2019 -> kept the same, not altered, shows experience       
- avg_activity_pace -> converted to avg_tb_pace (tb = training block)         
- fastest_pace -> the smallest value is kept              
- slowest_pace -> the largest value is kept             
- min_duration -> the smallest value is kept             
- max_duration -> the largest value is kept             
- min_distance -> the smallest value is kept             
- max_distance -> the largest value is kept             
- active_streak -> converted to: total_active_days             
- rest_streak -> converted to: total_rest_days               
- avg_race_pace -> kept the same, not altered, it is a target value            
- marathon_race_time -> kept the same, not altered, it is a target value

## 16-Week Dataframe

## 14-Week Dataframe

# Data Preprocessing

# Model Creation, Training & Evaluation